# Mortality pooling: 1 versus 1,000 policyholders

This notebook recreates the experiment behind Figure 6.1: distributions of
dynamic hedging costs for PE, TI and EI contracts on the same carbon-aware fund.
It uses the current library's costs discounted to time zero; positive cost
means additional funding. Dynamic hedging uses the selected discrete grid.
Published legacy cost values are not numerical acceptance targets.

Every experiment input appears below, with one explanation cell followed by
one Python cell. The displayed model values are explicit paper-calibration
starting values, not a claim to reproduce a separate edited notebook.
No preset configuration is loaded. Initial premiums and deltas are computed
with the chosen initial valuation settings.

Save under `CarbonFund_UL/notebooks/`, select the `.venv-analysis` kernel,
and run the cells in order. The existing C++ executable must be built with
OpenMP enabled to use multiple outer threads. `RUN_SIMULATIONS=False` reloads
saved CSVs; use the same simulation inputs and run directory when reloading.
Use a new `RUN_NAME` for each changed simulation experiment.


In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import math
import sys

PROJECT_ROOT = Path('/home/edoardolombardo/Documents/codes/CarbonFund_UL')
BINARY = PROJECT_ROOT / 'build-paper-release' / 'carbonfund_experiment'
OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'policyholder_pooling'
RUN_NAME = 'cir_T20_1_vs_1000_01'
RUN_SIMULATIONS = True
OVERWRITE_EXISTING = False
SAVE_FIGURES = True
SHOW_FIGURES = True

sys.path.insert(0, str(PROJECT_ROOT / 'python'))
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from IPython.display import HTML, display
import carbonfund_experiments as cfe

RUN_DIRECTORY = OUTPUT_ROOT / RUN_NAME
if RUN_SIMULATIONS and not BINARY.is_file():
    raise FileNotFoundError(f'Build carbonfund_experiment first: {BINARY}')
print('Python module:', Path(cfe.__file__).resolve())
print('Output directory:', RUN_DIRECTORY)


**Market inputs.** There are exactly four stocks. Returns and volatilities are
annualized; the risk-free rate is continuously compounded. The correlation
matrix must be symmetric and positive definite with unit diagonal. The code
constructs covariance from volatilities and correlations. These are the
full-precision legacy correlations; Table 4.1 prints rounded entries.


In [ ]:
market = {
    'risk_free_rate': 0.05,
    'expected_returns': [0.25, 0.15, 0.10, 0.08],
    'volatilities': [0.30, 0.25, 0.20, 0.16],
    'correlations': [
        [1.0000, 0.4397, 0.3900, 0.3168],
        [0.4397, 1.0000, 0.2954, 0.3261],
        [0.3900, 0.2954, 1.0000, 0.3134],
        [0.3168, 0.3261, 0.3134, 1.0000],
    ],
}


**Carbon inputs.** The paper uses four CIR intensities,
$dC_i=\kappa_i(\theta_i-C_i)dt+\sigma_i\sqrt{C_i}\,dB_i$,
simulated by the library's CIR scheme. All four entries can be changed.
The other supported model names are `OU` and `GeoOU`.
OU uses the same parameter keys with additive noise. GeoOU models the log
intensity as OU; replace `long_run_mean` with `long_run_log_mean` when selecting
it, and keep initial intensities positive. For CIR, initial and long-run
intensities are nonnegative. Stocks share the chosen market correlations;
the current carbon components and financial innovations are independent.


In [ ]:
carbon = {
    'model': 'CIR',
    'initial': [5000.0, 4000.0, 3000.0, 1000.0],
    'long_run_mean': [2500.0, 2000.0, 1500.0, 500.0],
    'mean_reversion': [0.05, 0.05, 0.05, 0.05],
    'volatility': [3.0, 3.0, 3.0, 3.0],
}


**Fund inputs.** Risk aversion and carbon penalization determine the fund's
stock weights from the observed carbon state. Larger carbon penalization
discourages exposure to carbon-intensive stocks. These preferences and the
market/carbon inputs remain common to all portfolio sizes and contract types.


In [ ]:
fund = {
    'risk_aversion': 1.0,
    'carbon_penalty': 0.0025,
}


**Contract inputs.** The benefit at elapsed time $t$ is
$\min(U_0e^{g_Ut},\max(L_0e^{g_Lt},X_t))$ per unit of notional.
PE pays on survival at maturity; TI pays on death before maturity; EI combines
survival payment and a death payment multiplied by the death-benefit fraction.
These are nominal benefits, discounted once in present-value costs.
Guarantees stay anchored to these original terms when the observed fund changes.
`fund_value` is the initial unit fund value, not the portfolio's total notional.
The growth rates are explicit inputs; changing the market rate does not
automatically change them.


In [ ]:
contract = {
    'fund_value': 1.0,
    'maturity': 20.0,
    'minimum_guarantee': 1.0,
    'maximum_guarantee': 1.0,
    'minimum_guarantee_growth_rate': 0.05,
    'maximum_guarantee_growth_rate': 0.50,
    'endowment_death_benefit_fraction': 1.0,
}


**Mortality inputs.** For a policyholder aged $a$ initially,
$\mu(a+t)=\xi+b^{-1}\exp((a+t-m)/b)$.
This uses the corrected minus sign and $\xi=0.0041959$.
The scale is `b`, not its reciprocal. Lifetimes are independent across
policyholders under this deterministic mortality law; there is no common
stochastic mortality factor in this experiment.


In [ ]:
mortality = {
    'xi': 0.0041959,
    'm': 79.6921211,
    'b': 11.5818911,
}


**Population and exposure inputs.** Select any of PE, TI and EI and the sizes
to compare. The defaults are one and 1,000 homogeneous lives, all aged 60.
Holding age fixed isolates the effect of pooling mortality risk.

`fixed_total` splits a fixed total notional across $n$ lives, so the default
notionals are 1 for one life and 0.001 for each of 1,000 lives.
`fixed_per_policy` assigns the same notional to each life; aggregate exposure
then grows with $n$. With the default analysis units `per_notional`, both
conventions report comparable costs per unit of total exposure.
Aggregate costs under `fixed_per_policy` also reflect the larger exposure.


In [ ]:
CONTRACT_TYPES = ['PE', 'TI', 'EI']
PORTFOLIO_SIZES = [1, 1000]
POLICYHOLDER_AGE = 60.0
EXPOSURE_MODE = 'fixed_total'       # 'fixed_total' or 'fixed_per_policy'
TOTAL_NOTIONAL = 1.0               # Used by fixed_total
NOTIONAL_PER_POLICY = 1.0          # Used by fixed_per_policy

if (not CONTRACT_TYPES or len(set(CONTRACT_TYPES)) != len(CONTRACT_TYPES)
        or any(kind not in ('PE', 'TI', 'EI') for kind in CONTRACT_TYPES)):
    raise ValueError('Select distinct PE, TI and/or EI contract types.')
if (1 not in PORTFOLIO_SIZES or len(set(PORTFOLIO_SIZES)) != len(PORTFOLIO_SIZES)
        or any(isinstance(n, bool) or not isinstance(n, int) or n < 1
               for n in PORTFOLIO_SIZES)):
    raise ValueError('Use distinct positive integer sizes and include 1 as the reference.')
if EXPOSURE_MODE not in ('fixed_total', 'fixed_per_policy'):
    raise ValueError('Unknown exposure mode.')
if any(not math.isfinite(x) or x <= 0 for x in (TOTAL_NOTIONAL, NOTIONAL_PER_POLICY)):
    raise ValueError('Notional inputs must be finite and positive.')

populations = {}
for kind in CONTRACT_TYPES:
    for n in PORTFOLIO_SIZES:
        q = TOTAL_NOTIONAL / n if EXPOSURE_MODE == 'fixed_total' else NOTIONAL_PER_POLICY
        populations[(kind, n)] = [
            {'type': kind, 'age': POLICYHOLDER_AGE, 'notional': q}
            for _ in range(n)
        ]
        print(kind, 'lives:', n, 'notional per life:', q, 'total:', n * q)


**Outer simulation inputs.** Each scenario has one common carbon/fund path
shared by the insured lives and independent mortality draws for those lives.
The same market seeds and grid are reused across all experiments, so the
market and carbon scenarios agree across portfolio sizes and contract types.
`threads=16` parallelizes scenarios in C++; zero uses the OpenMP runtime limit.
`progress_every=200` prints each 200 completed scenarios and the final count.
The paper reports 10,000 scenarios; the editable budget below starts from
your recent 5,000-scenario choice.

These scenario sizes are not antithetic: each CSV row is one independent outer
scenario. Antithetic sampling, if selected below, is inside valuation.


In [ ]:
outer_simulation = {
    'outer_scenarios': 5000,
    'intervals': 1040,
    'progress_every': 200,
    'threads': 16,
    'carbon_seed': 2026101801,
    'financial_seed': 2026101802,
    'mortality_seed': 2026101803,
}


**Initial valuation inputs.** These settings compute the initial premium and
delta of each portfolio at time zero, once per experiment. They do not simulate
actual deaths. All runs use the same initial seeds; for homogeneous lives,
initial prices and deltas per unit notional should agree across sizes.
Available configurations are `standard`, `standard_antithetic`, `conditioned`
and `conditioned_antithetic`. Antithetic `paths=4000` means 2,000 independent
pair averages; odd path requests round up to complete pairs.


In [ ]:
initial_valuation = {
    'paths': 4000,
    'intervals': 4000,
    'configuration': 'conditioned_antithetic',
    'carbon_seed': 2026101804,
    'financial_seed': 2026101805,
}


**Revaluation inputs.** Reprice the surviving liabilities and compute deltas
at each intermediate outer date. The number of inner intervals is
$\max(N_{\min},\lceil N_{\mathrm{base}}(T-t)/T\rceil)$.
`minimum_intervals` must be between 1 and `intervals`; making both 256 fixes
the count at 256. These are inner pricing intervals, independent of the outer
hedge dates. The configuration below keeps your recent fixed-grid comparison
explicit. The minimum setting requires the updated experiment interface.
Choose a new output run name after changing it.


In [ ]:
revaluation = {
    'paths': 40,
    'intervals': 256,
    'minimum_intervals': 256,
    'configuration': 'conditioned_antithetic',
    'carbon_seed': 2026101806,
    'financial_seed': 2026101807,
}


**Analysis inputs.** `per_notional` divides each cost by the portfolio's total
notional and is the default for the pooling comparison. `aggregate` reports
total costs; `per_policy` reports the cost divided by the number of lives.
For the fixed-total convention, `per_policy` changes the exposure unit with
population size, so it is not the default here.
Variance uses `ddof=1`; mean SE is sample standard deviation divided by the
square root of the number of outer scenarios. Mean intervals are conditional
on the computed initial premium. Quantiles and upper-tail expected shortfalls
are in the same selected cost units. The comparison family size below is an
explicit Bonferroni budget for strategy contrasts across the six default runs;
it does not adjust the individual mean intervals or the pooling variance ratios.


In [ ]:
analysis = {
    'normalization': 'per_notional',
    'confidence_level': 0.95,
    'comparison_family_size': 18,
    'quantiles': [0.025, 0.50, 0.90, 0.95, 0.975, 0.99],
    'expected_shortfall_levels': [0.95, 0.99],
    'bins': 100,
    'zoom': None,
    'plot_range': None,
    'figure_formats': ['png', 'pdf'],
    'dpi': 180,
    'title': 'Mortality pooling: 1 versus 1000 policyholders',
}
DISPLAY_DECIMALS = 4


**Figure inputs.** The main density figure follows the paper's three-panel
layout when all three contract types are selected. Its default histograms use
all observations and integrate to one for each population; each population
has its own bin width over its full data range. KDE is an optional smooth
alternative, using `scott`, `silverman` or a positive numeric bandwidth factor.
A constant sample is drawn as a point mass rather than given a fictitious PDF.
Optional x limits crop the view without renormalizing densities or statistics.
The same normalization is used for statistics and every figure.


In [ ]:
plot_settings = {
    'strategy': 'dynamic_hedge',   # Also 'no_hedge' or 'static_hedge'
    'density_method': 'histogram',  # 'histogram' or 'kde'
    'density_grid_points': 2048,
    'kde_bandwidth': 'scott',
    'histogram_alpha': 0.50,
    'line_width': 1.8,
    'figure_size': [12.0, 8.0],
    'x_limits': {'PE': None, 'TI': None, 'EI': None},
    'colors': ['#56B4E9', '#D55E00', '#009E73', '#CC79A7', '#E69F00'],
    'boxplot_show_fliers': True,
    'boxplot_whiskers': 1.5,
    'show_inverse_sqrt_reference': True,
}
if plot_settings['strategy'] not in cfe.STRATEGIES:
    raise ValueError('Unknown plot strategy.')
if plot_settings['density_method'] not in ('histogram', 'kde'):
    raise ValueError('Unknown density method.')


**Assemble and validate the configurations.** Each contract/size combination
gets a separate complete configuration. Changing an earlier input requires
rerunning this cell before running simulations or reloading analyses.
The printed audit records exposure, numerical budgets and normalization.
There are six experiments for the defaults, each using all three hedging
strategies on common outcomes. No model data are loaded from hidden presets.


In [ ]:
configs = {}
for key, people in populations.items():
    cfg = {
        'schema_version': 1,
        'experiment': 'hedging',
        'market': deepcopy(market),
        'carbon': deepcopy(carbon),
        'fund': deepcopy(fund),
        'contract': deepcopy(contract),
        'mortality': deepcopy(mortality),
        'population': deepcopy(people),
        'hedging': {
            **deepcopy(outer_simulation),
            'initial': deepcopy(initial_valuation),
            'revaluation': deepcopy(revaluation),
        },
        'analysis': deepcopy(analysis),
        'notes': 'Figure 6.1 comparison; shared market streams; corrected time-zero hedging costs.',
    }
    cfe.validate_config(cfg)
    configs[key] = cfg

audit = {
    'contract_types': CONTRACT_TYPES, 'population_sizes': PORTFOLIO_SIZES,
    'age': POLICYHOLDER_AGE, 'exposure_mode': EXPOSURE_MODE,
    'total_notional_input': TOTAL_NOTIONAL, 'notional_per_policy_input': NOTIONAL_PER_POLICY,
    'normalization': analysis['normalization'],
    'outer_simulation': outer_simulation,
    'initial_valuation': initial_valuation, 'revaluation': revaluation,
}
print(json.dumps(audit, indent=2))


**Run or reload.** C++ writes the raw scenario costs, premium, trading gains,
claims and terminal fund values. Python writes each run's summary and paired
strategy comparisons. All raw monetary CSV values are aggregate values;
normalization is applied during analysis. Nothing is kept only in notebook RAM.

To reuse completed runs, set `RUN_SIMULATIONS=False` and keep the simulation
inputs and run name unchanged. Analysis and plot settings can change. This
reload route regenerates summaries and figures without calling C++.
Full carbon/fund path histories are not exported; the market streams are
reproducible from the configuration, seeds and backend version.


In [ ]:
def simulation_inputs(cfg):
    value = deepcopy({k: v for k, v in cfg.items() if k not in ('analysis', 'notes')})
    value['hedging']['revaluation'].setdefault('minimum_intervals', 1)
    return value

results = {}
for (kind, n), cfg in configs.items():
    destination = RUN_DIRECTORY / kind / f'n_{n}'
    print(f'\n{kind}, {n} lives -> {destination}', flush=True)
    if RUN_SIMULATIONS:
        result = cfe.run_experiment(
            cfg, destination, binary=BINARY, overwrite=OVERWRITE_EXISTING,
            plots=False, progress=True,
        )
    else:
        result = cfe.load_result(destination)
        if simulation_inputs(result.config) != simulation_inputs(cfg):
            raise ValueError(f'{kind}/{n}: saved and current simulation inputs differ.')
        result.config['analysis'] = deepcopy(analysis)
        result.save_analysis(plots=False)
    results[(kind, n)] = result
    run = result.metadata['run']
    print('Initial price:', run['initial_price'])
    print('Actual outer threads:', run['threads_used'])
    print('Outer elapsed seconds:', run['simulation_wall_seconds'])

RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
manifest = {
    **audit, 'model_inputs': {k: deepcopy(configs[next(iter(configs))][k])
                            for k in ('market', 'carbon', 'fund', 'contract', 'mortality')},
    'analysis': deepcopy(analysis), 'plot_settings': deepcopy(plot_settings),
    'runs': [{'contract': kind, 'policyholders': n,
              'directory': str(result.directory.relative_to(RUN_DIRECTORY))}
             for (kind, n), result in results.items()],
}
(RUN_DIRECTORY / 'pooling_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')


**Check the comparison.** All runs must have exactly the same scenario IDs and
terminal fund values. Together with the common grid and stream schedule, this
checks reuse of market scenarios. Initial price and delta per unit notional
should agree across sizes because the initial ages and per-unit benefits agree.

Within a fixed size, mortality streams agree across PE/TI/EI, enabling the EI
additivity check. Across different sizes, the current sequential mortality
sampler does not make the single life a nested first life of the larger
portfolio: extra mortality draws change later positions in the stream. Each
run still has independent individual lifetimes with the correct law. The
distribution comparison does not require nested mortality paths.


In [ ]:
def read_raw(result):
    with (result.directory / 'hedging.csv').open(newline='') as stream:
        rows = list(csv.DictReader(stream))
    return {key: np.array([float(row[key]) for row in rows]) for key in rows[0]}

raw = {key: read_raw(result) for key, result in results.items()}
first = next(iter(raw.values()))
for value in raw.values():
    np.testing.assert_array_equal(value['scenario_id'], first['scenario_id'])
    np.testing.assert_array_equal(value['fund_terminal'], first['fund_terminal'])

for kind in CONTRACT_TYPES:
    reference = results[(kind, 1)]
    base_price = reference.metadata['run']['initial_price'] / reference.metadata['total_notional']
    base_delta = np.array(reference.metadata['run']['initial_deltas_PE_TI_EI']) / reference.metadata['total_notional']
    for n in PORTFOLIO_SIZES:
        item = results[(kind, n)]
        total = item.metadata['total_notional']
        np.testing.assert_allclose(item.metadata['run']['initial_price'] / total,
                                   base_price, rtol=1e-10, atol=1e-12)
        np.testing.assert_allclose(np.array(item.metadata['run']['initial_deltas_PE_TI_EI']) / total,
                                   base_delta, rtol=1e-10, atol=1e-12)

if set(CONTRACT_TYPES) == {'PE', 'TI', 'EI'}:
    rho = contract['endowment_death_benefit_fraction']
    for n in PORTFOLIO_SIZES:
        for field in ('claims_pv', 'initial_price', *cfe.STRATEGIES):
            target = raw[('PE', n)][field] + rho * raw[('TI', n)][field]
            np.testing.assert_allclose(raw[('EI', n)][field], target,
                rtol=1e-10, atol=1e-12 * max(1.0, float(np.max(np.abs(target)))))
print('Shared market outcomes, initial unit valuations and available EI identities: verified.')


**Complete statistics.** The combined CSV includes each contract, size and
hedging strategy, with mean, variance, standard deviation, mean SE and interval,
extrema, probability of positive cost, quantiles and expected shortfall.
Floating statistics display in scientific notation with four decimals;
exported values retain full floating-point precision.


In [ ]:
def write_rows(path, rows):
    with Path(path).open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def show_rows(rows):
    def format_value(key, value):
        if isinstance(value, (float, np.floating)):
            return (f'{value:.4f}%' if key.endswith('_percent')
                    else f'{value:.{DISPLAY_DECIMALS}e}')
        return html.escape(str(value))
    heads = ''.join(f'<th>{html.escape(key)}</th>' for key in rows[0])
    body = ''.join('<tr>' + ''.join(f'<td>{format_value(k, v)}</td>'
                   for k, v in row.items()) + '</tr>' for row in rows)
    display(HTML('<div style="overflow:auto"><table><thead><tr>' + heads
                 + '</tr></thead><tbody>' + body + '</tbody></table></div>'))

summaries = {key: result.summary() for key, result in results.items()}
summary_rows = [
    {'contract': kind, 'policyholders': n,
     'total_notional': result.metadata['total_notional'],
     'normalization': analysis['normalization'], **row}
    for (kind, n), result in results.items() for row in summaries[(kind, n)]
]
show_rows(summary_rows)
write_rows(RUN_DIRECTORY / 'pooling_summary.csv', summary_rows)
print('Saved:', RUN_DIRECTORY / 'pooling_summary.csv')


**Quantify pooling.** Compare each population with the single-life baseline,
separately for each contract and strategy, using the same selected units.
Variance reduction is $100(1-s_n^2/s_1^2)$; the single-life row is zero percent.
This is a comparison of realized hedge costs, not the variance reduction of a
pricing estimator. Ratios are descriptive estimates without variance-ratio
confidence intervals. A near-zero single-life variance makes ratios undefined.
Common carbon, market and valuation errors do not automatically diversify
with the number of lives, so the total cost SD need not follow $n^{-1/2}$.


In [ ]:
summary_lookup = {
    (kind, n, row['strategy']): row
    for (kind, n), rows in summaries.items() for row in rows
}
pooling_rows = []
for kind in CONTRACT_TYPES:
    for strategy in cfe.STRATEGIES:
        baseline = summary_lookup[(kind, 1, strategy)]
        for n in PORTFOLIO_SIZES:
            row = summary_lookup[(kind, n, strategy)]
            ratio = row['variance'] / baseline['variance'] if baseline['variance'] > 0 else math.nan
            pooling_rows.append({
                'contract': kind, 'strategy': strategy, 'policyholders': n,
                'mean': row['mean'], 'variance': row['variance'], 'std': row['std'],
                'mean_se': row['mean_se'], 'mean_ci_low': row['mean_ci_low'],
                'mean_ci_high': row['mean_ci_high'],
                'variance_ratio_vs_one': ratio,
                'std_ratio_vs_one': math.sqrt(ratio),
                'variance_reduction_percent': 0.0 if n == 1 else 100.0 * (1.0 - ratio),
            })
selected_rows = [row for row in pooling_rows if row['strategy'] == plot_settings['strategy']]
show_rows(selected_rows)
write_rows(RUN_DIRECTORY / 'pooling_comparison.csv', pooling_rows)


**Density plots: the Figure 6.1 comparison.** Overlay the chosen strategy's
distributions for each population size. PE and TI are on the top row, with
EI centered below when all three types are selected. The x axis always names
the selected normalization. Positive values mean additional funding.
Histograms are estimates, not exact probability densities. A narrow pooled
distribution can have a much higher density peak while its area stays one.


In [ ]:
contract_labels = {'PE': 'Pure Endowment', 'TI': 'Term Insurance', 'EI': 'Endowment Insurance'}
strategy_labels = dict(zip(cfe.STRATEGIES, cfe.LABELS))
units = {'aggregate': 'Aggregate cost at time zero',
         'per_policy': 'Cost per policy at time zero',
         'per_notional': 'Cost per unit total notional at time zero'}
strategy_index = cfe.STRATEGIES.index(plot_settings['strategy'])
costs = {}
for (kind, n), result in results.items():
    denominator = {'aggregate': 1.0, 'per_policy': result.metadata['policy_count'],
                   'per_notional': result.metadata['total_notional']}[analysis['normalization']]
    costs[(kind, n)] = result.costs[:, strategy_index] / denominator
colors = {n: plot_settings['colors'][i % len(plot_settings['colors'])]
          for i, n in enumerate(PORTFOLIO_SIZES)}

def comparison_axes(title):
    fig = plt.figure(figsize=plot_settings['figure_size'], layout='constrained')
    if len(CONTRACT_TYPES) == 3:
        grid = fig.add_gridspec(2, 4)
        positions = (grid[0, :2], grid[0, 2:], grid[1, 1:3])
    else:
        grid = fig.add_gridspec(1, len(CONTRACT_TYPES))
        positions = [grid[0, i] for i in range(len(CONTRACT_TYPES))]
    axes = {kind: fig.add_subplot(position)
            for kind, position in zip(CONTRACT_TYPES, positions)}
    for kind, ax in axes.items():
        ax.set_title(contract_labels[kind])
        ax.set_xlabel(units[analysis['normalization']])
        ax.grid(alpha=0.20)
    fig.suptitle(title + ' | ' + strategy_labels[plot_settings['strategy']])
    return fig, axes

def finish_figure(fig, stem):
    if SAVE_FIGURES:
        for extension in analysis['figure_formats']:
            fig.savefig(RUN_DIRECTORY / f'{stem}.{extension}', dpi=analysis['dpi'])
    if SHOW_FIGURES:
        display(fig)
    plt.close(fig)

size_label = ' versus '.join(f'{n:,}' for n in PORTFOLIO_SIZES)
fig, axes = comparison_axes(size_label + ' lives: hedging-cost densities')
for kind, ax in axes.items():
    combined = np.concatenate([costs[(kind, n)] for n in PORTFOLIO_SIZES])
    limits = plot_settings['x_limits'].get(kind)
    lo, hi = (float(combined.min()), float(combined.max())) if limits is None else limits
    if lo == hi:
        lo, hi = lo - 0.5, hi + 0.5
    grid = np.linspace(lo, hi, plot_settings['density_grid_points'])
    for n in PORTFOLIO_SIZES:
        x = costs[(kind, n)]
        label = f'{n:,} policyholder' + ('' if n == 1 else 's')
        if np.ptp(x) == 0:
            ax.axvline(x[0], color=colors[n], label=label + ' (point mass)')
        elif plot_settings['density_method'] == 'histogram':
            ax.hist(x, bins=analysis['bins'], density=True, color=colors[n],
                    alpha=plot_settings['histogram_alpha'], label=label)
        else:
            density = gaussian_kde(x, bw_method=plot_settings['kde_bandwidth'])
            ax.plot(grid, density(grid), color=colors[n], label=label,
                    linewidth=plot_settings['line_width'])
    if limits is not None:
        ax.set_xlim(limits)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
finish_figure(fig, 'pooling_density')


**Empirical distribution functions.** These plots use the complete samples
without density smoothing or binning. A steeper CDF shows a more concentrated
distribution. Optional x limits only change the displayed range.


In [ ]:
fig, axes = comparison_axes('Mortality pooling: empirical distribution functions')
for kind, ax in axes.items():
    for n in PORTFOLIO_SIZES:
        ordered = np.sort(costs[(kind, n)])
        probabilities = np.arange(1, len(ordered) + 1) / len(ordered)
        ax.step(ordered, probabilities, where='post', color=colors[n],
                linewidth=plot_settings['line_width'], label=f'{n:,} lives')
    if plot_settings['x_limits'].get(kind) is not None:
        ax.set_xlim(plot_settings['x_limits'][kind])
    ax.set_ylabel('Empirical cumulative probability')
    ax.set_ylim(0.0, 1.02)
    ax.legend(fontsize=9)
finish_figure(fig, 'pooling_ecdf')


**Box plots.** Compare medians, interquartile ranges and tails in the same cost
units. The whisker multiplier and visibility of outliers are selectable above.
Hiding outliers affects only the plot, not statistics or saved raw data.


In [ ]:
fig, axes = comparison_axes('Mortality pooling: box plots')
for kind, ax in axes.items():
    boxes = ax.boxplot([costs[(kind, n)] for n in PORTFOLIO_SIZES],
                       patch_artist=True,
                       showfliers=plot_settings['boxplot_show_fliers'],
                       whis=plot_settings['boxplot_whiskers'])
    for patch, n in zip(boxes['boxes'], PORTFOLIO_SIZES):
        patch.set_facecolor(colors[n])
        patch.set_alpha(0.65)
    ax.set_xticks(range(1, len(PORTFOLIO_SIZES) + 1))
    ax.set_xticklabels([f'{n:,}' for n in PORTFOLIO_SIZES])
    ax.set_xlabel('Number of policyholders')
    ax.set_ylabel(units[analysis['normalization']])
    ax.axhline(0.0, color='black', linewidth=0.8, linestyle=':')
finish_figure(fig, 'pooling_boxplots')


**Dispersion versus portfolio size.** This plot is most informative after
adding intermediate sizes, for example `[1, 10, 100, 1000]`, in the population
input cell and running those experiments. The optional $n^{-1/2}$ line is
anchored at the single-life SD and represents a hypothetical case in which
all risk diversifies. It is not a prediction for the total cost: common carbon
risk, numerical hedging error and shared valuation noise can leave a floor.
All means, variances and figures can be recomputed from saved runs later.


In [ ]:
fig, axes = comparison_axes('Mortality pooling: standard deviation')
ordered_sizes = np.array(sorted(PORTFOLIO_SIZES))
for kind, ax in axes.items():
    sd = np.array([summary_lookup[(kind, int(n), plot_settings['strategy'])]['std']
                   for n in ordered_sizes])
    ax.plot(ordered_sizes, sd, marker='o', linewidth=plot_settings['line_width'],
            color='#0072B2', label='Observed cost SD')
    if plot_settings['show_inverse_sqrt_reference']:
        reference = summary_lookup[(kind, 1, plot_settings['strategy'])]['std'] / np.sqrt(ordered_sizes)
        ax.plot(ordered_sizes, reference, linestyle='--', color='#777777',
                label='All-risk-diversifiable reference')
    ax.set_xscale('log')
    if np.all(sd > 0):
        ax.set_yscale('log')
    ax.set_xlabel('Number of policyholders')
    ax.set_ylabel('Standard deviation in selected cost units')
    ax.legend(fontsize=8)
finish_figure(fig, 'pooling_dispersion')
print('Completed analysis in:', RUN_DIRECTORY)
